# GCP Pose Estimation — Kaggle Training

## Before running: download the data as a ZIP and upload to Kaggle

**1. Download the Drive folder as ZIP (on your laptop)**
- Open [drive.google.com](https://drive.google.com) → find the shared GCP data folder
- Right-click the folder → **Download**
- Chrome will zip it and download. If the folder is >2 GB, Drive splits it into multiple ZIPs — download **all** of them.

**2. Upload the ZIP(s) as a Kaggle dataset**
- kaggle.com → your profile → **Datasets** → **New Dataset**
- Upload all ZIP file(s) → title: `skylark-gcp-data` → visibility: **Private** → Create

**3. Add that dataset as input to this notebook**
- In this notebook → **Add Input** (top right) → Your Datasets → `skylark-gcp-data` → Add

Then run cells top to bottom. GPU T4×2 + Internet must both be ON.

In [1]:
# 1. Clone repo + install dependencies
!git clone https://github.com/J4KE-B/skylark-gcp gcp
%cd gcp
!pip -q install -r requirements.txt

Cloning into 'gcp'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 145 (delta 62), reused 125 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 55.82 KiB | 6.20 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/kaggle/working/gcp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you 

In [2]:
# 2. Wire config to Kaggle input dataset
import os, json, yaml, glob
from pathlib import Path

hits = sorted(glob.glob('/kaggle/input/**/gcp_marks.json', recursive=True), key=len)
assert hits, 'gcp_marks.json not found — add skylark-gcp-data as notebook input'
label_path = hits[0]
BASE = os.path.dirname(label_path)
print('Dataset root:', BASE)
print('Contents:', sorted(os.listdir(BASE)))

# Load + patch labels first; we use a real label key to locate the image root
os.makedirs('data', exist_ok=True)
raw = json.load(open(label_path))
patched = {k.replace('&', 'and'): v for k, v in raw.items()}
local_labels = 'data/gcp_marks.json'
json.dump(patched, open(local_labels, 'w'))
n_fixed = sum(1 for k in raw if '&' in k)
if n_fixed: print(f'Patched {n_fixed} label keys (& -> and)')

def find_dir(base, *names):
    for n in names:
        p = os.path.join(base, n)
        if os.path.isdir(p): return p
    raise FileNotFoundError(f'None of {names} in {base}')

def resolve_by_label(base, label_keys):
    """Find the dir that actually holds the labelled site folders, regardless of wrapper
    levels (train_clean/train/train_dataset/...) or junk dirs (__MACOSX). Locates the site
    folder of a sample label key and returns its parent."""
    site = next(iter(label_keys)).split('/')[0]
    for p in Path(base).rglob(site):
        if p.is_dir() and '__MACOSX' not in p.parts:
            return str(p.parent)
    return base

def descend_to_images(base, max_depth=8):
    """Test set (no labels): descend, skipping junk, until a folder holds images or branches."""
    junk = {'__MACOSX', '.ipynb_checkpoints'}
    d = base
    for _ in range(max_depth):
        entries = [e for e in os.listdir(d) if e not in junk]
        subs = [e for e in entries if os.path.isdir(os.path.join(d, e))]
        has_img = any(f.lower().endswith(('.jpg', '.jpeg')) for f in entries)
        if has_img or len(subs) != 1:
            return d
        d = os.path.join(d, subs[0])
    return d

train_base = find_dir(BASE, 'train_clean', 'train_dataset', 'train')
test_base  = find_dir(BASE, 'test_clean',  'test_dataset',  'test')
train_dir  = resolve_by_label(train_base, patched)
test_dir   = descend_to_images(test_base)
print(f'train_dir : {train_dir}')
print(f'test_dir  : {test_dir}')

cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['paths'] = {'label_file': local_labels, 'train_dir': train_dir,
                'test_dir': test_dir, 'output_dir': 'outputs'}
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'))
print('Config updated.')

labels = json.load(open(local_labels))
found  = sum(1 for k in labels if os.path.exists(os.path.join(train_dir, k)))
pct    = 100 * found // len(labels)
print(f'Train images on disk: {found}/{len(labels)} ({pct}%)')
print('OK — enough to train.' if found >= len(labels) * 0.8 else
      'WARNING: <80% images — check folder names above.')

Dataset root: /kaggle/input/datasets/jacobantonyjeejo/skylark-gcp-data
Contents: ['gcp_marks.json', 'test_clean', 'train_clean']
train_dir : /kaggle/input/datasets/jacobantonyjeejo/skylark-gcp-data/train_clean/train/train_dataset
test_dir  : /kaggle/input/datasets/jacobantonyjeejo/skylark-gcp-data/test_clean/test_dataset
Config updated.
Train images on disk: 1000/1000 (100%)
OK — enough to train.


In [3]:
# 3. SMOKE: 3 epochs, then inspect predictions schema before committing GPU time
import yaml
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 3
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import json; p = json.load(open('predictions.json'))
k = next(iter(p)); print(k, '->', p[k]); print('total:', len(p))

device: cuda
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 169MB/s]
Ep  1 val loss=1.063 PCK10=0.000 PCK25=0.007 F1=0.941
  -> saved best (score=0.961)
Ep  2 val loss=0.951 PCK10=0.033 PCK25=0.105 F1=0.964
  -> saved best (score=1.273)
Ep  3 val loss=0.856 PCK10=0.026 PCK25=0.184 F1=0.940
  -> saved best (score=1.421)
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
infer: 100%|████████████████████████████████████| 25/25 [00:31<00:00,  1.26s/it]
wrote 300 predictions -> predictions.json
231129_CTD/231129_CTD_GDA94/11/DJI_20231129142314_01

In [4]:
# 4. FULL run: 60 epochs (early-stops on patience=15)
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 60
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml

device: cuda
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
Ep  1 val loss=1.156 PCK10=0.007 PCK25=0.020 F1=0.953
  -> saved best (score=1.071)
Ep  2 val loss=1.026 PCK10=0.020 PCK25=0.059 F1=0.941
  -> saved best (score=1.145)
Ep  3 val loss=0.853 PCK10=0.039 PCK25=0.158 F1=0.929
  -> saved best (score=1.389)
Ep  4 val loss=1.070 PCK10=0.053 PCK25=0.138 F1=0.918
Ep  5 val loss=1.032 PCK10=0.059 PCK25=0.217 F1=0.898
  -> saved best (score=1.457)
Ep  6 val loss=1.013 PCK10=0.072 PCK25=0.184 F1=0.952
  -> saved best (score=1.478)
Ep  7 val loss=0.789 PCK10=0.053 PCK25=0.237 F1=0.969
  -> saved best (score=1.587)
Ep  8 val loss=0.921 PCK10=0.072 PCK25=0.191 F1=0.946
Ep  9 val loss=0.880 PCK10=0.105 PCK25=0.243 F1=0.942
Ep 10 val loss=0.937 PCK10=0.105 PCK25=0.270 F1=0.947
  -> saved best (score=1.605)
Ep 11 val loss=1.090 PCK10=0.079 PCK25=0.309 F1=0.877
  -> saved

In [5]:
# 4b. Stage-2: train the crop shape-classifier (ResNet-18 on a high-res crop
# around the marker). The whole-image model can't read a ~35px marker — it's
# sub-pixel at the encoder stride — so shape is classified from a crop instead.
!python train_crop.py --config configs/config.yaml

device: cuda
crop train/val = 999/1 (10 train sites, 1 val sites, no overlap)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 195MB/s]
Ep  1 train_loss=0.483 f1=0.791 | val_loss=0.390 f1=0.333
  -> saved best_crop (val_f1=0.333)
Ep  2 train_loss=0.125 f1=0.953 | val_loss=0.072 f1=0.333
Ep  3 train_loss=0.076 f1=0.973 | val_loss=0.068 f1=0.333
Ep  4 train_loss=0.102 f1=0.981 | val_loss=0.193 f1=0.333
Ep  5 train_loss=0.066 f1=0.979 | val_loss=0.354 f1=0.333
Ep  6 train_loss=0.048 f1=0.983 | val_loss=0.003 f1=0.333
Ep  7 train_loss=0.031 f1=0.991 | val_loss=0.059 f1=0.333
Ep  8 train_loss=0.023 f1=0.992 | val_loss=0.005 f1=0.333
Ep  9 train_loss=0.028 f1=0.994 | val_loss=0.001 f1=0.333
Ep 10 train_loss=0.021 f1=0.994 | val_loss=0.003 f1=0.333
Ep 11 train_loss=0.010 f1=0.998 | val_loss=0.004 f1=0.333
Ep 12 train_loss=0.013 f1=0.997 | 

In [6]:
# 5. Final predictions (two-stage: keypoint model + high-res crop classifier)
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt \
    --crop-checkpoint outputs/best_crop.pt --output predictions.json
import pandas as pd; pd.read_csv('outputs/training_log.csv').tail()

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for keypoints, but no transform to process it.
  self._set_keys()
infer: 100%|████████████████████████████████████| 25/25 [00:32<00:00,  1.28s/it]
Stage 2: reclassifying shapes from high-res crops...
crop-cls: 100%|█████████████████████████████████| 10/10 [00:34<00:00,  3.46s/it]
wrote 300 predictions -> predictions.json


,epoch,train_loss,train_kp,train_cls,train_pck10,train_pck25,train_pck50,train_f1,val_loss,val_kp,val_cls,val_pck10,val_pck25,val_pck50,val_f1
37,38,0.510429,0.506373,0.008113,0.155660,0.575472,0.801887,0.997106,1.010800,0.847088,0.327423,0.118421,0.381579,0.506579,0.963731
38,39,0.504917,0.499076,0.011682,0.168632,0.575472,0.797170,0.995069,0.979400,0.859037,0.240727,0.085526,0.309211,0.460526,0.932558
39,40,0.516432,0.512581,0.007703,0.148585,0.591981,0.790094,0.996889,1.000285,0.870331,0.259907,0.125000,0.296053,0.440789,0.946381
40,41,0.512409,0.509643,0.005532,0.150943,0.568396,0.774764,0.996071,1.129840,0.918102,0.423477,0.098684,0.328947,0.473684,0.922882
41,42,0.523162,0.520784,0.004755,0.188679,0.602594,0.783019,0.997095,1.039269,0.856938,0.364661,0.065789,0.368421,0.500000,0.952952


In [7]:
# 6. Visualize predictions at ORIGINAL resolution — hollow circle, split into parts
import os, json, glob, cv2, shutil, zipfile, math

pred_hits = sorted(glob.glob('/kaggle/working/**/predictions.json', recursive=True), key=len)
assert pred_hits, 'predictions.json not found — run the predict cell first'
preds = json.load(open(pred_hits[0]))
print(f'Loaded {len(preds)} predictions from {pred_hits[0]}')

def descend_to_images(base, max_depth=8):
    junk = {'__MACOSX', '.ipynb_checkpoints'}
    d = base
    for _ in range(max_depth):
        entries = [e for e in os.listdir(d) if e not in junk]
        subs = [e for e in entries if os.path.isdir(os.path.join(d, e))]
        has_img = any(f.lower().endswith(('.jpg', '.jpeg')) for f in entries)
        if has_img or len(subs) != 1:
            return d
        d = os.path.join(d, subs[0])
    return d

test_base = sorted(glob.glob('/kaggle/input/**/test_clean', recursive=True) +
                   glob.glob('/kaggle/input/**/test_dataset', recursive=True), key=len)[0]
test_dir = descend_to_images(test_base)
print('test_dir:', test_dir)

out_dir = '/kaggle/working/viz'
shutil.rmtree(out_dir, ignore_errors=True); os.makedirs(out_dir)

drawn = 0
for key, val in preds.items():
    p = os.path.join(test_dir, key)
    if not os.path.exists(p): continue
    img = cv2.imread(p)
    if img is None: continue
    h, w = img.shape[:2]
    x, y = int(val['mark']['x']), int(val['mark']['y'])
    s = val['verified_shape']
    r  = max(int(w * 0.03),  70)
    th = max(int(w * 0.003), 3)
    fs = max(w / 600.0, 1.5)
    cv2.circle(img, (x, y), r, (0, 0, 255), th)
    cv2.putText(img, f'{s} ({x},{y})', (x + r + 8, y - r - 8),
                cv2.FONT_HERSHEY_SIMPLEX, fs, (0, 255, 255), th)
    cv2.imwrite(os.path.join(out_dir, key.replace('/', '_')), img,
                [cv2.IMWRITE_JPEG_QUALITY, 100])
    drawn += 1

# Split the annotated images into PARTS so each zip is downloadable
PARTS = 4                                   # bump to 6/8 if each part is still too big
files = sorted(glob.glob(f'{out_dir}/*'))
per = math.ceil(len(files) / PARTS)
print(f'\nAnnotated {drawn} images -> splitting into {PARTS} zips:')
for i in range(PARTS):
    chunk = files[i*per:(i+1)*per]
    if not chunk: break
    zp = f'/kaggle/working/viz_part{i+1}.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_STORED) as z:  # JPEGs already compressed
        for f in chunk:
            z.write(f, os.path.basename(f))
    print(f'  {zp}: {len(chunk)} imgs, {os.path.getsize(zp)/1e6:.0f} MB')

Loaded 300 predictions from /kaggle/working/gcp/predictions.json
test_dir: /kaggle/input/datasets/jacobantonyjeejo/skylark-gcp-data/test_clean/test_dataset

Annotated 300 images -> splitting into 4 zips:
  /kaggle/working/viz_part1.zip: 75 imgs, 508 MB
  /kaggle/working/viz_part2.zip: 75 imgs, 551 MB
  /kaggle/working/viz_part3.zip: 75 imgs, 632 MB
  /kaggle/working/viz_part4.zip: 75 imgs, 606 MB


## Artifacts to download

From the Kaggle Output panel (right sidebar), download:

- `gcp/outputs/best.pt` — keypoint model weights (upload to Drive, copy link)
- `gcp/outputs/best_crop.pt` — stage-2 crop shape-classifier weights
- `gcp/outputs/training_log.csv` — keypoint metrics (PCK@10/25/50 + F1)
- `gcp/outputs/crop_log.csv` — crop classifier val F1 per epoch
- `gcp/predictions.json` — submission file (two-stage predictions)
- `gcp/viz_part*.zip` — annotated images (split into downloadable parts)
